In [ ]:
# ============================================================
# CREDORA FINANCE — Credit Risk & Loan Approval Analytics
# Notebook 02: Feature Engineering + Baseline Models (FULL)
# ============================================================
# HOW TO USE (IMPORTANT — read this first):
#   1. Paste each "# %%" block into its own Jupyter cell.
#   2. Press RUN ALL — do NOT run cells one by one out of order.
#      Every error you've hit was a fresh kernel missing earlier
#      cells. Run All fixes that permanently.
#   3. Prerequisite: setx MYSQL_PASSWORD "your_password" once in
#      Command Prompt, then restart VS Code/Jupyter.
# ============================================================

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, URL

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

sns.set_style("whitegrid")
RANDOM_STATE = 42
print("Imports OK")

In [ ]:
url = URL.create(
    drivername="mysql+mysqlconnector",
    username="root",
    password=os.environ["MYSQL_PASSWORD"],
    host="localhost",
    database="credora_risk"
)
engine = create_engine(url)

customers = pd.read_sql("SELECT * FROM customers", engine)
credit_history = pd.read_sql("SELECT * FROM credit_history", engine)
loans = pd.read_sql("SELECT * FROM loan_applications", engine)
transactions = pd.read_sql("SELECT * FROM transactions", engine)

print(f"Loaded: {len(customers):,} customers | {len(loans):,} loans | "
      f"{len(transactions):,} transactions")

In [ ]:
credit_amt = transactions.loc[transactions['transaction_type'] == 'Credit'] \
    .groupby('customer_id')['amount'].sum()
debit_amt = transactions.loc[transactions['transaction_type'] == 'Debit'] \
    .groupby('customer_id')['amount'].sum()
emi_amt = transactions.loc[transactions['category'] == 'EMI'] \
    .groupby('customer_id')['amount'].sum()

txn_agg = pd.DataFrame({
    'txn_count': transactions.groupby('customer_id').size(),
    'total_inflow': credit_amt,
    'total_outflow': debit_amt,
    'emi_outflow': emi_amt,
    'avg_balance': transactions.groupby('customer_id')['balance_after'].mean(),
}).fillna(0).reset_index()

txn_agg['net_cashflow'] = txn_agg['total_inflow'] - txn_agg['total_outflow']
txn_agg['inflow_outflow_ratio'] = (
    txn_agg['total_inflow'] / txn_agg['total_outflow'].replace(0, np.nan))

print(f"txn_agg: {txn_agg.shape[0]:,} customers × {txn_agg.shape[1]} features")

In [ ]:
model_df = (loans
            .merge(credit_history, on='customer_id', how='left')
            .merge(customers.drop(columns=['email', 'phone']),
                   on='customer_id', how='left')
            .merge(txn_agg, on='customer_id', how='left'))

model_df = model_df[model_df['disbursed_flag'] == 1].copy()
model_df['loan_to_income'] = (
    model_df['requested_amount'] / model_df['annual_income'].replace(0, np.nan))

print(f"Modeling rows (disbursed only): {len(model_df):,}")
print(f"Default rate: {model_df['loan_default'].mean()*100:.1f}%")
# Expected: 3,403 rows, ~23.4%

In [ ]:
# FIXED: application_id is kept as the DataFrame INDEX (not a feature
# column) instead of being dropped entirely. 04_risk_scoring later
# needs a real key to merge scored rows back onto the right customer -
# dropping the ID completely made that impossible to do safely.
model_df = model_df.set_index('application_id')

LEAKAGE_COLS = ['approval_status', 'approved_amount', 'decision_date',
                'disbursed_flag', 'customer_id',
                'credit_id', 'first_name', 'last_name',
                'date_of_birth', 'customer_since', 'as_of_date',
                'application_date']

model_df = model_df.drop(columns=[c for c in LEAKAGE_COLS
                                  if c in model_df.columns])
print(f"Remaining columns: {model_df.shape[1]}")
print(f"Index is now application_id: {model_df.index.name}")

In [ ]:
y = model_df['loan_default'].astype(int)
X = model_df.drop(columns=['loan_default'])

cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
print(f"Categorical ({len(cat_cols)}): {cat_cols}")
print(f"Numeric ({len(num_cols)}): {num_cols}")

X[num_cols] = X[num_cols].fillna(X[num_cols].median())
X[cat_cols] = X[cat_cols].fillna('Unknown')
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print(f"Final feature matrix: {X.shape[0]:,} rows × {X.shape[1]} features")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"Train default rate: {y_train.mean()*100:.1f}% | "
      f"Test default rate: {y_test.mean()*100:.1f}%")

In [ ]:
def evaluate(model, X_tr, X_te, name):
    model.fit(X_tr, y_train)
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]
    metrics = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1': f1_score(y_test, pred),
        'ROC-AUC': roc_auc_score(y_test, proba),
    }
    print(f"\n===== {name} =====")
    for k, v in metrics.items():
        if k != 'Model':
            print(f"{k:>10}: {v:.3f}")
    cm = confusion_matrix(y_test, pred)
    ConfusionMatrixDisplay(cm, display_labels=['Repaid', 'Default']).plot(
        cmap='Blues', values_format='d')
    plt.title(f'{name} — Confusion Matrix')
    plt.show()
    return metrics

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(class_weight='balanced', max_iter=1000,
                        random_state=RANDOM_STATE)
lr_metrics = evaluate(lr, X_train_s, X_test_s, 'Logistic Regression')

In [ ]:
dt = DecisionTreeClassifier(max_depth=5, class_weight='balanced',
                            random_state=RANDOM_STATE)
dt_metrics = evaluate(dt, X_train, X_test, 'Decision Tree')

plt.figure(figsize=(18, 6))
plot_tree(dt, feature_names=X.columns, class_names=['Repaid', 'Default'],
          filled=True, max_depth=2, fontsize=9)
plt.title('Decision Tree — top 2 levels')
plt.tight_layout()
plt.show()

In [ ]:
results = pd.DataFrame([lr_metrics, dt_metrics]).set_index('Model')
print("\n===== BASELINE COMPARISON =====")
print(results.round(3))

results[['Recall', 'ROC-AUC']].plot(kind='bar', edgecolor='white')
plt.title('Baseline Models — Recall & ROC-AUC')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# index=True is deliberate here — application_id must survive
# the round-trip to CSV so 03_modeling/04_risk_scoring can key
# scored rows back to the right customer later.
X.assign(loan_default=y.values).to_csv('model_ready.csv', index=True)
results.round(4).to_csv('baseline_results.csv')
print(f"Saved model_ready.csv: {X.shape[0]:,} rows × {X.shape[1]} features")
print("Saved baseline_results.csv")

In [ ]:
print("""
WEEK 2 COMPLETE when:
[ ] txn_agg = 5,000 rows (one per customer)
[ ] Modeling table = 3,403 disbursed rows, default rate ~23.4%
[ ] Leakage columns dropped
[ ] Train/test default rates both ~23%
[ ] Both baselines show Recall + ROC-AUC
[ ] model_ready.csv + baseline_results.csv saved

NEXT: 03_modeling_and_risk_scoring.py (Random Forest + XGBoost +
SHAP + 0-100 risk score + scored_customers table).
""")